In [3]:
%run experiment_parser.ipynb

import json

with open("setup_parse.ipynb", "r") as file:
    setup_notebook = json.load(file)

for cell in setup_notebook["cells"]:
    if cell["cell_type"] == "code":
        code = "".join(cell["source"])
        definitions = code.split('setup_parser("./test_files/setup_valid.json")')[0]
        exec(definitions)

%run quantum_gate_algorithm_library.ipynb

from qiskit import QuantumCircuit, transpile

In [4]:
def testing_framework(algorithm, setup_json_path, experiment_json_path):
    """
    @brief Prepares, runs, and evaluates an algorithm on the Aer simulator.

    @param algorithm Function taking no arguments and returning an unmeasured
                     circuit. Initial state preparation belongs to the setup.
    @param setup_json_path Path to the setup JSON file.
    @param experiment_json_path Path to the experiment JSON file.
    @return Dictionary with the circuit, backend name, counts, expected output,
            shots, observed accuracy, required accuracy, and pass/fail result.
    @exception TypeError Raised for an invalid algorithm or initial circuit.
    @exception ValueError Raised for incompatible or unsupported test inputs.
    @note Expected output must be a full binary measurement string. Qubit zero
          is the rightmost bit. Oracles and hardware execution are unsupported.
          Errors from the parsers and simulator propagate to the caller.
    """
    if not callable(algorithm):
        raise TypeError("algorithm must be a function.")

    backend, shots, accuracy = experiment_parser(experiment_json_path)
    qubit_count, initial_circuits, oracle, expected_output = setup_parser(setup_json_path)

    if backend.name != "aer_simulator":
        raise ValueError("This framework currently supports only aer_simulator.")
    if oracle is not None:
        raise ValueError("Oracle integration is not supported yet.")
    if not isinstance(expected_output, str):
        raise ValueError("Expected output must be a binary string, such as 01.")
    if len(expected_output) != qubit_count or any(bit not in "01" for bit in expected_output):
        raise ValueError("Expected output must contain one binary digit per qubit.")
    if len(initial_circuits) != qubit_count:
        raise ValueError("Setup must provide one initial circuit per qubit.")

    algorithm_circuit = algorithm()
    if not isinstance(algorithm_circuit, QuantumCircuit):
        raise TypeError("The algorithm must return a QuantumCircuit.")
    if algorithm_circuit.num_qubits != qubit_count:
        raise ValueError("The algorithm and setup must use the same number of qubits.")
    if algorithm_circuit.num_clbits != 0:
        raise ValueError("The algorithm must not contain classical bits or measurements.")

    circuit = QuantumCircuit(qubit_count)
    for index, initial_circuit in enumerate(initial_circuits):
        if not isinstance(initial_circuit, QuantumCircuit):
            raise TypeError("The setup parser must return QuantumCircuit objects for initial states.")
        circuit.compose(initial_circuit, qubits=[index], inplace=True)
    circuit.compose(algorithm_circuit, inplace=True)
    circuit.measure_all()

    compiled_circuit = transpile(circuit, backend)
    job = backend.run(compiled_circuit, shots=shots)
    counts = job.result().get_counts()

    matching_shots = counts.get(expected_output, 0)
    observed_accuracy = matching_shots / sum(counts.values())

    return {
        "status": "completed",
        "circuit": circuit,
        "backend": backend.name,
        "shots": shots,
        "counts": counts,
        "expected_output": expected_output,
        "observed_accuracy": observed_accuracy,
        "required_accuracy": accuracy,
        "passed": observed_accuracy >= accuracy
    }

In [5]:
result = testing_framework(
    x_circuit,
    "test_files/setup_x_zero_to_one.json",
    "test_files/experiment_one_qubit.json"
)

print("Counts:", result["counts"])
print("Expected output:", result["expected_output"])
print("Observed accuracy:", result["observed_accuracy"])
print("Required accuracy:", result["required_accuracy"])
print("PASS" if result["passed"] else "FAIL")

Counts: {'1': 1000}
Expected output: 1
Observed accuracy: 1.0
Required accuracy: 0.9
PASS
